In [1]:
# ==========================================
# 【Cell 1】后端基建：纯净版双擎模型、进程守护与内网穿透 (终极免依赖版)
# ==========================================
import os
import subprocess
import time
import re
import urllib.request
from IPython.display import display, Javascript

# ------------------------------------------
# ⚙️ 核心全局配置区
# ------------------------------------------
OLLAMA_PORT = "11434"
OLLAMA_CMD = "/usr/local/bin/ollama" # 绝对路径防迷路

# 🌟 听你的，保留指定的模型版本
MODELS_TO_PULL = [
    "qwen3.5:9b",
    "qwen3-vl:8b"
]

print("🧹 正在清理旧的后台进程，确保端口通畅...")
os.system("pkill -9 -f 'ollama serve'")
os.system("pkill -9 -f 'cloudflared'")
time.sleep(2)

print("📦 正在安装核心底层依赖库 (已彻底剥离 Python 库，物理防错)...")
!nvidia-smi
# 👇 补全 zstd 解压工具
!apt-get update && apt-get install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

# 防掉线守护进程
display(Javascript('''
    console.log("✅ Colab 防掉线守护进程已启动！");
    function KeepClicking() {
        let btn = document.querySelector("colab-connect-button");
        if (btn) { btn.click(); }
    }
    setInterval(KeepClicking, 60000);
'''))

# ------------------------------------------
# 🚀 临时启动 Ollama 引擎与穿透隧道
# ------------------------------------------
print("🔄 启动临时 Ollama 核心引擎并开放跨域...")
env = os.environ.copy()
env["OLLAMA_ORIGINS"] = "*"
env["OLLAMA_HOST"] = "0.0.0.0"
temp_server = subprocess.Popen([OLLAMA_CMD, "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("⏳ 等待 Ollama 服务就绪...")
ready = False
for _ in range(15):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{OLLAMA_PORT}/", timeout=1)
        ready = True
        break
    except:
        time.sleep(1)

if not ready:
    print("❌ Ollama 启动失败，请检查资源。")
else:
    print("✅ Ollama 引擎运转正常！")

print("🔗 正在配置 Cloudflare 纯净隧道...")
cf_process = subprocess.Popen(
    ["./cloudflared-linux-amd64", "tunnel", "--url", f"http://127.0.0.1:{OLLAMA_PORT}"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

api_url = None
cf_logs = []

# 👇 扩大 Cloudflare 的日志读取范围至 150 行
for _ in range(150):
    line = cf_process.stdout.readline()
    if not line:
        break
    cf_logs.append(line.strip())
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        api_url = match.group(0)
        break

if api_url:
    print(f"\n" + "="*60)
    print(f"✅ 【你的专属云端 API 地址】: {api_url}")
    print(f"👉 如果使用 Cherry Studio，请将此地址填入 Base URL")
    print("="*60 + "\n")
else:
    print("\n❌ 未能获取到 Cloudflare 节点地址！请看底层诊断日志：")
    print("\n".join(cf_logs[-15:]))

# ------------------------------------------
# 🧠 并发拉取多模态模型组 (恢复网速和进度条显示！)
# ------------------------------------------
print("📥 正在为系统招募双模态专家 (体积较大，请耐心等待)...")
for model in MODELS_TO_PULL:
    print(f"   ► 正在原生拉取模型: {model} ...")
    # 👇 核心修改：用 ! 魔法指令替代 os.system，完美呼出带速度的进度条！
    !{OLLAMA_CMD} pull {model}
    print(f"   ✔️ {model} 部署完成！")

print("\n🚀 所有底层引擎均已挂载就绪！")

# ==========================================
# 🛑 核心修改：关闭后台，切换为【前台明牌启动】阻塞主线程
# ==========================================
print("\n🛑 正在将后台引擎切换至前台，准备打印心电图日志...\n")
temp_server.terminate()
os.system("pkill -9 -f 'ollama serve'")
time.sleep(2)

# 👇 最末尾追加前台启动指令，锁死主线程防掉线
!OLLAMA_ORIGINS="*" OLLAMA_HOST="0.0.0.0" {OLLAMA_CMD} serve